# Debug: Instance Type Matching Investigation

This notebook investigates why instance types from `instance_rates` don't match `vm_costs`.


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_INSTANCE_RATES = f"{CATALOG}.{SCHEMA}.instance_rates"
TABLE_VM_COSTS = f"{CATALOG}.{SCHEMA}.vm_costs"

print(f"✅ Tables: {TABLE_INSTANCE_RATES}, {TABLE_VM_COSTS}")


In [ ]:
# Check what's in each table
print("=" * 60)
print("📊 TABLE ROW COUNTS")
print("=" * 60)

print("\n📋 instance_rates (DBU rates):")
display(spark.sql(f"""
    SELECT cloud, COUNT(*) as count 
    FROM {TABLE_INSTANCE_RATES} 
    GROUP BY cloud
"""))

print("\n📋 vm_costs (VM pricing):")
display(spark.sql(f"""
    SELECT cloud, pricing_tier, COUNT(*) as count 
    FROM {TABLE_VM_COSTS} 
    GROUP BY cloud, pricing_tier
    ORDER BY cloud, pricing_tier
"""))


In [ ]:
# ============================================================
# AZURE: Compare instance_type formats
# ============================================================
print("=" * 60)
print("🔍 AZURE: Instance Type Comparison")
print("=" * 60)

print("\n📋 Sample instance_rates (DBU table) - AZURE:")
display(spark.sql(f"""
    SELECT DISTINCT instance_type 
    FROM {TABLE_INSTANCE_RATES} 
    WHERE cloud = 'AZURE'
    ORDER BY instance_type
    LIMIT 20
"""))

print("\n📋 Sample vm_costs (VM pricing) - AZURE:")
display(spark.sql(f"""
    SELECT DISTINCT instance_type 
    FROM {TABLE_VM_COSTS} 
    WHERE cloud = 'AZURE'
    ORDER BY instance_type
    LIMIT 20
"""))


In [ ]:
# Check exact matches vs mismatches
print("=" * 60)
print("🔍 AZURE: Matching Analysis")
print("=" * 60)

# Get lists
dbu_types = spark.sql(f"""
    SELECT DISTINCT instance_type FROM {TABLE_INSTANCE_RATES} WHERE cloud = 'AZURE'
""").toPandas()['instance_type'].tolist()

vm_types = spark.sql(f"""
    SELECT DISTINCT instance_type FROM {TABLE_VM_COSTS} WHERE cloud = 'AZURE'
""").toPandas()['instance_type'].tolist()

print(f"\n📊 DBU table has {len(dbu_types)} unique Azure instance types")
print(f"📊 VM costs table has {len(vm_types)} unique Azure instance types")

# Find exact matches
exact_matches = set(dbu_types) & set(vm_types)
print(f"\n✅ Exact matches: {len(exact_matches)}")
if exact_matches:
    print(f"   Examples: {list(exact_matches)[:5]}")

# Find in DBU but not in VM
dbu_only = set(dbu_types) - set(vm_types)
print(f"\n❌ In DBU table but NOT in vm_costs: {len(dbu_only)}")
if dbu_only:
    print(f"   Examples: {list(dbu_only)[:10]}")

# Find in VM but not in DBU
vm_only = set(vm_types) - set(dbu_types)
print(f"\n⚠️ In vm_costs but NOT in DBU table: {len(vm_only)}")
if vm_only:
    print(f"   Examples: {list(vm_only)[:10]}")


In [ ]:
# Try to find patterns in naming differences
print("=" * 60)
print("🔍 AZURE: Naming Pattern Analysis")
print("=" * 60)

# Normalize and try to match
dbu_normalized = {}
for t in dbu_types:
    # Various normalization attempts
    dbu_normalized[t.lower()] = t
    dbu_normalized[t.lower().replace("standard_", "")] = t
    dbu_normalized[t.lower().replace("_", " ")] = t

vm_normalized = {}
for t in vm_types:
    vm_normalized[t.lower()] = t
    vm_normalized[t.lower().replace("standard_", "")] = t
    vm_normalized[f"standard_{t.lower()}"] = t

# Check normalized matches
norm_matches = set(dbu_normalized.keys()) & set(vm_normalized.keys())
print(f"\n✅ Normalized matches found: {len(norm_matches)}")

# Show specific examples of what should match
print("\n📋 Side-by-side comparison (first 15):")
print(f"{'DBU Table':<30} | {'VM Costs Table':<30}")
print("-" * 62)
for i, (d, v) in enumerate(zip(sorted(dbu_types)[:15], sorted(vm_types)[:15])):
    print(f"{d:<30} | {v:<30}")


In [ ]:
# ============================================================
# AWS: Same analysis for AWS
# ============================================================
print("=" * 60)
print("🔍 AWS: Instance Type Comparison")
print("=" * 60)

print("\n📋 Sample instance_rates (DBU table) - AWS:")
display(spark.sql(f"""
    SELECT DISTINCT instance_type 
    FROM {TABLE_INSTANCE_RATES} 
    WHERE cloud = 'AWS'
    ORDER BY instance_type
    LIMIT 15
"""))

print("\n📋 Sample vm_costs (VM pricing) - AWS:")
display(spark.sql(f"""
    SELECT DISTINCT instance_type 
    FROM {TABLE_VM_COSTS} 
    WHERE cloud = 'AWS'
    ORDER BY instance_type
    LIMIT 15
"""))


In [ ]:
# ============================================================
# GCP: Same analysis for GCP
# ============================================================
print("=" * 60)
print("🔍 GCP: Instance Type Comparison")
print("=" * 60)

print("\n📋 Sample instance_rates (DBU table) - GCP:")
display(spark.sql(f"""
    SELECT DISTINCT instance_type 
    FROM {TABLE_INSTANCE_RATES} 
    WHERE cloud = 'GCP'
    ORDER BY instance_type
    LIMIT 15
"""))

print("\n📋 Sample vm_costs (VM pricing) - GCP:")
display(spark.sql(f"""
    SELECT DISTINCT instance_type 
    FROM {TABLE_VM_COSTS} 
    WHERE cloud = 'GCP'
    ORDER BY instance_type
    LIMIT 15
"""))


In [ ]:
# ============================================================
# DETAILED AZURE MISMATCH ANALYSIS
# ============================================================
print("=" * 60)
print("🔍 AZURE: Detailed Mismatch Analysis")
print("=" * 60)

# Get all Azure instance types from both tables
dbu_azure = spark.sql(f"""
    SELECT DISTINCT instance_type FROM {TABLE_INSTANCE_RATES} WHERE cloud = 'AZURE'
""").toPandas()['instance_type'].tolist()

vm_azure = spark.sql(f"""
    SELECT DISTINCT instance_type FROM {TABLE_VM_COSTS} WHERE cloud = 'AZURE'
""").toPandas()['instance_type'].tolist()

# Find missing
missing = sorted(set(dbu_azure) - set(vm_azure))

print(f"\n📊 DBU table: {len(dbu_azure)} Azure instance types")
print(f"📊 VM costs: {len(vm_azure)} Azure instance types")
print(f"❌ Missing from VM costs: {len(missing)}")

# Show ALL missing instance types
print(f"\n📋 ALL missing instance types ({len(missing)}):")
for m in missing[:50]:  # Show first 50
    print(f"   - {m}")

if len(missing) > 50:
    print(f"   ... and {len(missing) - 50} more")


In [ ]:
# Group missing by prefix to see patterns
print("=" * 60)
print("🔍 Missing Instance Types by Prefix/Pattern")
print("=" * 60)

from collections import Counter

def get_prefix(inst):
    """Extract prefix pattern from instance type"""
    inst = str(inst).replace('Standard_', '')
    # Get first part before underscore or digits
    import re
    match = re.match(r'^([A-Za-z]+)', inst)
    return match.group(1) if match else 'Unknown'

# Count by prefix
prefix_counts = Counter(get_prefix(m) for m in missing)
print("\n📊 Missing by instance family prefix:")
for prefix, count in sorted(prefix_counts.items(), key=lambda x: -x[1]):
    print(f"   {prefix}: {count}")

# Also check what's in VM costs
vm_prefix_counts = Counter(get_prefix(v) for v in vm_azure)
print("\n📊 Available in VM costs by prefix:")
for prefix, count in sorted(vm_prefix_counts.items(), key=lambda x: -x[1]):
    print(f"   {prefix}: {count}")


In [ ]:
# Check if there are similar names that ALMOST match
print("=" * 60)
print("🔍 Near-Match Analysis (fuzzy matching)")
print("=" * 60)

# For each missing, find closest match in vm_costs
def find_similar(missing_inst, vm_list, max_results=3):
    """Find similar instance names"""
    missing_lower = missing_inst.lower().replace('standard_', '')
    similar = []
    for v in vm_list:
        v_lower = v.lower().replace('standard_', '')
        # Check if one contains the other, or starts same
        if missing_lower[:5] == v_lower[:5]:
            similar.append(v)
    return similar[:max_results]

print("\n📋 Examples of near-matches:")
for m in missing[:20]:
    similar = find_similar(m, vm_azure)
    if similar:
        print(f"   {m}")
        print(f"      → Similar in VM costs: {similar}")
    else:
        print(f"   {m}")
        print(f"      → No similar found")


In [ ]:
# Check what instance types the Azure VM notebook actually fetched
print("=" * 60)
print("🔍 VM Costs Table - What was actually fetched?")
print("=" * 60)

# Show all unique Azure instance types from vm_costs
print(f"\n📋 ALL Azure instance types in vm_costs ({len(vm_azure)}):")
for v in sorted(vm_azure):
    print(f"   - {v}")


In [ ]:
# ============================================================
# QUERY AZURE API DIRECTLY TO SEE WHAT'S AVAILABLE
# ============================================================
print("=" * 60)
print("🔍 Direct Azure API Query - What instance types are available?")
print("=" * 60)

import requests

# Query Azure API for eastus region to see what's available
region = "eastus"
base_url = "https://prices.azure.com/api/retail/prices"
filter_query = (
    f"serviceName eq 'Virtual Machines' "
    f"and armRegionName eq '{region}' "
    f"and priceType eq 'Consumption' "
    f"and contains(productName, 'Windows') eq false"
)

params = {"api-version": "2023-01-01-preview", "$filter": filter_query}

# Get all unique armSkuName values from first few pages
api_instance_types = set()
next_page = base_url
page_count = 0

print(f"\n🔄 Querying Azure API for {region}...")

while next_page and page_count < 10:  # Limit to 10 pages for speed
    response = requests.get(base_url if page_count == 0 else next_page, 
                           params=params if page_count == 0 else None, timeout=30)
    data = response.json()
    
    for item in data.get('Items', []):
        sku_name = item.get('armSkuName', '')
        if sku_name:
            api_instance_types.add(sku_name)
    
    next_page = data.get('NextPageLink')
    page_count += 1

print(f"✅ Found {len(api_instance_types)} unique instance types in Azure API")

# Show sample
print(f"\n📋 Sample Azure API instance types (first 30):")
for t in sorted(api_instance_types)[:30]:
    print(f"   - {t}")


In [ ]:
# ============================================================
# COMPARE: What's in DBU table vs What Azure API has
# ============================================================
print("=" * 60)
print("🔍 DBU Table vs Azure API Comparison")
print("=" * 60)

# Normalize both for comparison
dbu_normalized = set(t.lower() for t in dbu_azure)
api_normalized = set(t.lower() for t in api_instance_types)

# Find matches
matches = dbu_normalized & api_normalized
dbu_only = dbu_normalized - api_normalized
api_only = api_normalized - dbu_normalized

print(f"\n📊 Results:")
print(f"   DBU table (normalized): {len(dbu_normalized)}")
print(f"   Azure API (normalized): {len(api_normalized)}")
print(f"   ✅ Matches: {len(matches)}")
print(f"   ❌ In DBU but NOT in API: {len(dbu_only)}")
print(f"   ⚠️ In API but NOT in DBU: {len(api_only)}")

# Show examples of what's missing from API
print(f"\n📋 Instance types in DBU but NOT found in Azure API (first 30):")
for t in sorted(dbu_only)[:30]:
    print(f"   - {t}")


In [ ]:
# ============================================================
# CHECK FOR NEAR-MATCHES (naming differences)
# ============================================================
print("=" * 60)
print("🔍 Near-Match Analysis: DBU vs API naming")
print("=" * 60)

# For each DBU instance that didn't match, find closest API name
def find_api_similar(dbu_inst, api_set):
    """Find similar names in API"""
    dbu_clean = dbu_inst.replace('standard_', '')
    similar = []
    for api_inst in api_set:
        api_clean = api_inst.replace('standard_', '')
        # Check if base name (before _v) is similar
        dbu_base = dbu_clean.split('_v')[0] if '_v' in dbu_clean else dbu_clean
        api_base = api_clean.split('_v')[0] if '_v' in api_clean else api_clean
        if dbu_base == api_base or dbu_base in api_clean or api_base in dbu_clean:
            similar.append(api_inst)
    return similar[:3]

print("\n📋 Examples where DBU name differs from API name:")
count = 0
for dbu in sorted(dbu_only)[:50]:
    similar = find_api_similar(dbu, api_normalized)
    if similar:
        print(f"   DBU: {dbu}")
        print(f"   API: {similar}")
        print()
        count += 1
        if count >= 15:
            break

if count == 0:
    print("   No near-matches found - these instance types don't exist in Azure API")
